In [1]:
import numpy as np
import pandas as pd
import glob
from helper import *
from define_model import *
from my_helpers import *
import pprint as p
from sklearn.model_selection import train_test_split
from itertools import zip_longest



In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
p.pprint(tf.test.is_built_with_cuda())
print('*'*50)
p.pprint(tf.config.list_physical_devices('GPU'))
print('*'*50)
p.pprint(tf.test.gpu_device_name())

True
**************************************************
[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
**************************************************
'/device:GPU:0'


2023-07-26 21:39:06.934560: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:936] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2023-07-26 21:39:06.941336: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:936] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2023-07-26 21:39:06.941676: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:936] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2023-07-26 21:39:06.942352: I tensorflow/core/platform/cpu_feature_guard.cc:151] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags

In [4]:
# Below identifiers correspond to evaluation protocol 1, 2 and 3 respectively
# identifiers - "TouchPose_crossSession", "TouchPose_crossSubjects", "TouchPose_crossGestures"
identifier_eth = "TouchPose_crossGestures"
data_path_eth = "../data/eth_data/"
data_path_chili= "../data/processed/"
model_path = "../models/"

In [5]:
files = glob.glob(data_path_chili+'*.pkl')
chili_data = pd.concat([pd.read_pickle(fp) for fp in files], ignore_index=True)
chili_data['skeleton']= chili_data['skeleton'].apply(lambda x : x.astype(np.float32))
chili_data

,Timestamp,skeleton,ScanSizeX,ScanSizeY,trial,gesture,subject,day,cap_img
0,1.680093e+12,"[[-270.0, 288.0, 68.0], [-242.0, 276.0, 55.0],...",72,41,1,UP,32,4,"[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,..."
1,1.680093e+12,"[[-270.0, 288.0, 68.0], [-243.0, 276.0, 54.0],...",72,41,1,UP,32,4,"[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,..."
2,1.680093e+12,"[[-270.0, 288.0, 67.0], [-243.0, 276.0, 54.0],...",72,41,1,UP,32,4,"[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,..."
3,1.680093e+12,"[[-270.0, 288.0, 67.0], [-243.0, 276.0, 54.0],...",72,41,1,UP,32,4,"[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,..."
4,1.680093e+12,"[[-270.0, 288.0, 67.0], [-242.0, 276.0, 54.0],...",72,41,1,UP,32,4,"[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,..."
...,...,...,...,...,...,...,...,...,...
24028,1.680267e+12,"[[-280.0, 292.0, 156.0], [-265.0, 278.0, 126.0...",72,41,6,UP,13,1,"[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,..."
24029,1.680267e+12,"[[-281.0, 293.0, 155.0], [-266.0, 280.0, 124.0...",72,41,6,UP,13,1,"[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,..."
24030,1.680267e+12,"[[-281.0, 293.0, 154.0], [-266.0, 280.0, 123.0...",72,41,6,UP,13,1,"[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,..."
24031,1.680267e+12,"[[-282.0, 294.0, 153.0], [-267.0, 281.0, 122.0...",72,41,6,UP,13,1,"[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,..."


# Load partially model

In [6]:
model_eth = get_eth_model('TouchPose_crossGestures',65)
model = create_SkeletonModel(41, 72, 1)

2023-07-26 21:39:09.145914: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:936] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2023-07-26 21:39:09.146297: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:936] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2023-07-26 21:39:09.146580: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:936] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2023-07-26 21:39:09.146905: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:936] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2023-07-26 21:39:09.147305: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:936] successful NUMA node read from S

In [7]:
print(model_eth.input)
print(model_eth.output)

KerasTensor(type_spec=TensorSpec(shape=(None, 41, 72, 1), dtype=tf.float32, name='input_25'), name='input_25', description="created by layer 'input_25'")
KerasTensor(type_spec=TensorSpec(shape=(None, 41, 72, 1), dtype=tf.float32, name=None), name='regression_depth/BiasAdd:0', description="created by layer 'regression_depth'")


In [8]:
print(model.input)
print(model.output)

KerasTensor(type_spec=TensorSpec(shape=(None, 41, 72, 1), dtype=tf.float32, name='input_1'), name='input_1', description="created by layer 'input_1'")
KerasTensor(type_spec=TensorSpec(shape=(None, 63), dtype=tf.float32, name=None), name='regression/BiasAdd:0', description="created by layer 'regression'")


In [9]:
print(len(model.layers))

18


In [10]:
print(len(model_eth.layers))

26


In [11]:
match = []
for depth_layer, skel_layer, pos in zip(model_eth.layers,model.layers,range(len(model.layers))):
    match.append((depth_layer.name , skel_layer.name, pos ))
match

[('input_25', 'input_1', 0),
 ('conv2d_336', 'conv2d', 1),
 ('conv2d_337', 'conv2d_1', 2),
 ('max_pooling2d_72', 'max_pooling2d', 3),
 ('conv2d_338', 'conv2d_2', 4),
 ('conv2d_339', 'conv2d_3', 5),
 ('max_pooling2d_73', 'max_pooling2d_1', 6),
 ('conv2d_340', 'conv2d_4', 7),
 ('conv2d_341', 'conv2d_5', 8),
 ('max_pooling2d_74', 'max_pooling2d_2', 9),
 ('conv2d_342', 'conv2d_6', 10),
 ('conv2d_343', 'flatten', 11),
 ('up_sampling2d_72', 'dense', 12),
 ('concatenate_72', 'activation', 13),
 ('conv2d_344', 'batch_normalization', 14),
 ('conv2d_345', 'dropout', 15),
 ('up_sampling2d_73', 'dense_1', 16),
 ('concatenate_73', 'regression', 17)]

In [12]:
match = [ (''.join([ xi for xi in x[0] if not (xi.isdigit() or xi== '_')]),''.join([ xi for xi in x[1] if not (xi.isdigit() or xi== '_')]),x[2]) for x in match]
match

[('input', 'input', 0),
 ('convd', 'convd', 1),
 ('convd', 'convd', 2),
 ('maxpoolingd', 'maxpoolingd', 3),
 ('convd', 'convd', 4),
 ('convd', 'convd', 5),
 ('maxpoolingd', 'maxpoolingd', 6),
 ('convd', 'convd', 7),
 ('convd', 'convd', 8),
 ('maxpoolingd', 'maxpoolingd', 9),
 ('convd', 'convd', 10),
 ('convd', 'flatten', 11),
 ('upsamplingd', 'dense', 12),
 ('concatenate', 'activation', 13),
 ('convd', 'batchnormalization', 14),
 ('convd', 'dropout', 15),
 ('upsamplingd', 'dense', 16),
 ('concatenate', 'regression', 17)]

In [13]:
good_match = [x for x in match if x[0]==x[1]]
good_match

[('input', 'input', 0),
 ('convd', 'convd', 1),
 ('convd', 'convd', 2),
 ('maxpoolingd', 'maxpoolingd', 3),
 ('convd', 'convd', 4),
 ('convd', 'convd', 5),
 ('maxpoolingd', 'maxpoolingd', 6),
 ('convd', 'convd', 7),
 ('convd', 'convd', 8),
 ('maxpoolingd', 'maxpoolingd', 9),
 ('convd', 'convd', 10)]

In [14]:
for i in range(len(good_match)):
    pos = match[i][2]
    model.layers[pos].set_weights(model_eth.layers[pos].get_weights())

In [15]:
def data_splits(df):
    X_cap, Y_skeleton = None, None
    x_cap_img = np.array(df["cap_img"].tolist())
    y_skeleton = np.array(df["skeleton"].tolist()).reshape(-1,63)

    x_cap_img[x_cap_img < 0] = 0
    maxVal = 3023
    x_cap_img = x_cap_img/maxVal

    if X_cap is None:
        X_cap = x_cap_img
        Y_skeleton = y_skeleton
    else:
        X_cap = np.vstack((X_cap, x_cap_img))
        Y_skeleton = np.vstack((Y_skeleton, y_skeleton))
    return X_cap, Y_skeleton

X,y = data_splits(chili_data)

# Split data into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Define custom loss function
def custom_loss(y_true, y_pred):
    J = tf.constant(21.0)  # number of predicted joints
    return tf.reduce_sum(tf.abs(y_true - y_pred)) / (J * 3)

# Compile model
model.compile(optimizer='adam', loss=custom_loss, metrics=['accuracy'])

# Train model
history = model.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=50, batch_size=32)


Epoch 1/50


2023-07-26 21:39:40.079704: I tensorflow/stream_executor/cuda/cuda_dnn.cc:368] Loaded cuDNN version 8204


601/601 [==============================] - 14s 20ms/step - loss: 1945.2935 - accuracy: 0.2980 - val_loss: 1113.8998 - val_accuracy: 0.3911
Epoch 2/50
601/601 [==============================] - 12s 19ms/step - loss: 929.1737 - accuracy: 0.3673 - val_loss: 638.3380 - val_accuracy: 0.3865
Epoch 3/50
601/601 [==============================] - 12s 19ms/step - loss: 792.9661 - accuracy: 0.4156 - val_loss: 453.9507 - val_accuracy: 0.4839
Epoch 4/50
601/601 [==============================] - 12s 19ms/step - loss: 687.2848 - accuracy: 0.4325 - val_loss: 367.6413 - val_accuracy: 0.4315
Epoch 5/50
601/601 [==============================] - 12s 20ms/step - loss: 641.4562 - accuracy: 0.4293 - val_loss: 358.4830 - val_accuracy: 0.4760
Epoch 6/50
601/601 [==============================] - 12s 20ms/step - loss: 590.4966 - accuracy: 0.4449 - val_loss: 318.4039 - val_accuracy: 0.4396
Epoch 7/50
601/601 [==============================] - 12s 20ms/step - loss: 561.9404 - accuracy: 0.4436 - val_loss: 324.7

# Model from Scratch

In [90]:
model = create_SkeletonModel(41, 72, 1)

In [93]:
def data_splits(df):
    X_cap, Y_skeleton = None, None
    x_cap_img = np.array(df["cap_img"].tolist())
    y_skeleton = np.array(df["skeleton"].tolist()).reshape(-1,63)

    x_cap_img[x_cap_img < 0] = 0
    maxVal = 3023
    x_cap_img = x_cap_img/maxVal

    if X_cap is None:
        X_cap = x_cap_img
        Y_skeleton = y_skeleton
    else:
        X_cap = np.vstack((X_cap, x_cap_img))
        Y_skeleton = np.vstack((Y_skeleton, y_skeleton))
    return X_cap, Y_skeleton

X,y = data_splits(chili_data)

# Split data into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Define custom loss function
def custom_loss(y_true, y_pred):
    J = tf.constant(21.0)  # number of predicted joints
    return tf.reduce_sum(tf.abs(y_true - y_pred)) / (J * 3)

# Compile model
model.compile(optimizer='adam', loss=custom_loss, metrics=['accuracy'])

# Train model
history = model.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=50, batch_size=32)


Epoch 1/50
601/601 [==============================] - 14s 21ms/step - loss: 4893.0786 - accuracy: 0.0118 - val_loss: 1763.0599 - val_accuracy: 0.0064
Epoch 2/50
601/601 [==============================] - 12s 19ms/step - loss: 915.0121 - accuracy: 0.2943 - val_loss: 610.5091 - val_accuracy: 0.3894
Epoch 3/50
601/601 [==============================] - 12s 20ms/step - loss: 727.5671 - accuracy: 0.4171 - val_loss: 525.2785 - val_accuracy: 0.4383
Epoch 4/50
601/601 [==============================] - 12s 20ms/step - loss: 710.4316 - accuracy: 0.4443 - val_loss: 578.4828 - val_accuracy: 0.4117
Epoch 5/50
601/601 [==============================] - 12s 20ms/step - loss: 689.7516 - accuracy: 0.4538 - val_loss: 515.1827 - val_accuracy: 0.4042
Epoch 6/50
601/601 [==============================] - 12s 20ms/step - loss: 646.4268 - accuracy: 0.4712 - val_loss: 504.7592 - val_accuracy: 0.5315
Epoch 7/50
601/601 [==============================] - 12s 20ms/step - loss: 623.9100 - accuracy: 0.4668 - val_

: 

In [61]:
control = 'gesture'
chili_group = chili_data.groupby([control])[["cap_img","skeleton"]]
chili_id = list(chili_group.groups.keys())
p.pprint(control)
p.pprint(chili_group)
p.pprint(chili_id) 

'gesture'
['DQ', 'DT', 'LQ', 'LT', 'UP']


In [65]:
def data_splits(df, id_arr):
    X_cap, Y_skeleton = None, None
    for p_id in id_arr:
        df_extract = df.get_group(p_id)
        x_cap_img = np.array(df_extract["cap_img"].tolist())
        y_skeleton = np.array(df_extract["skeleton"].tolist()).reshape(-1,63)

        x_cap_img[x_cap_img < 0] = 0
        maxVal = 3023
        x_cap_img = x_cap_img/maxVal

        if X_cap is None:
            X_cap = x_cap_img
            Y_skeleton = y_skeleton
        else:
            X_cap = np.vstack((X_cap, x_cap_img))
            Y_skeleton = np.vstack((Y_skeleton, y_skeleton))
    return X_cap, Y_skeleton

def custom_joint_loss(y_true, y_pred):
    J = tf.constant(21.0)  # number of predicted joints
    loss = tf.reduce_mean(tf.square(y_true - y_pred))
    return (1.0 / (J * 3.0)) * tf.reduce_sum(loss)

skeleton_err = []


for i in range(len(chili_id)):
    test_set = [chili_id[i]]
    X_test_cap, Y_test_skeleton= data_splits(chili_group,test_set)
    print(X_test_cap[0].shape, Y_test_skeleton[0].shape)
    Y_out_skeleton = model.predict(X_test_cap, batch_size=32)
    skeleton_error = np.mean(abs(Y_test_skeleton - Y_out_skeleton[0])) 
    skeleton_err.append(skeleton_error)
    print("Accuracy", chili_id[i],skeleton_error)


(41, 72) (63,)
Accuracy DQ 210.80259836737005
(41, 72) (63,)
Accuracy DT 217.08282863679105
(41, 72) (63,)
Accuracy LQ 202.59242353931347
(41, 72) (63,)
Accuracy LT 196.89154848110707
(41, 72) (63,)
Accuracy UP 207.9491955273898
